# Sprint 2 - Preprocessing & Format Conversion (SEntFiN v1.1)

**Input:** `data/processed/sentfin_long.csv` (output Sprint 1)

**Keputusan yang dipegang di notebook ini:**
1. **Split di level headline (`s_no`), bukan per-pasangan** -> cegah kebocoran title yang sama ke train & test (67 duplikat + multi-entity).
2. **Stratifikasi pakai majority-sentiment per headline** -> aman dari bucket langka; distribusi entity-level diverifikasi setelah split.
3. **NER = word-level BIO (opsi A)**, single entity-type `ENT` (dataset tidak memberi tipe entitas). Opsi B (subword) dipertimbangkan nanti saat fine-tuning transformer.

**Output:**
- `ner_{train,val,test}.jsonl` -> `{s_no, tokens, ner_tags}`
- `absa_{train,val,test}.csv` -> `s_no, title, entity, sentiment, label`

In [1]:
import re
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT   = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

long_df = pd.read_csv(DATA_PROCESSED / 'sentfin_long.csv', encoding='utf-8')
print('Loaded long-format:', long_df.shape)
print('Headlines unik:', long_df['s_no'].nunique())
long_df.head()

Loaded long-format: (14409, 5)
Headlines unik: 10753


,s_no,title,entity,sentiment,exact_substring
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet,neutral,True
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC,neutral,True
2,3,"Mid-cap funds can deliver more, stay put: Experts",Mid-cap funds,positive,True
3,4,Mid caps now turn into market darlings,Mid caps,positive,True
4,5,"Market seeing patience, if not conviction: Pra...",Market,neutral,True


## 1. Investigasi duplikat title

Sprint 1 menemukan 67 title duplikat. Pertanyaan kritis: apakah duplikat itu **identik** (title + anotasi sama) atau **konflik** (title sama, anotasi beda)?

- Identik -> aman, cukup keep satu.
- Konflik -> berbahaya kalau bocor antar-split; harus diputuskan.

Kita bikin *signature* per headline = himpunan pasangan `(entity, sentiment)`, lalu bandingkan antar headline ber-title sama.

In [2]:
# Signature per headline: frozenset of (entity, sentiment)
sig = (long_df.groupby('s_no')
              .apply(lambda g: frozenset(zip(g['entity'], g['sentiment'])))
              .rename('signature'))
head = long_df.groupby('s_no')['title'].first().to_frame().join(sig)

# Title yang dipakai >1 s_no
dup_titles = head['title'].value_counts()
dup_titles = dup_titles[dup_titles > 1]
print(f'Title yang muncul di >1 headline: {len(dup_titles)}')

identical, conflicting = [], []
for t in dup_titles.index:
    grp = head[head['title'] == t]
    if grp['signature'].nunique() == 1:
        identical.append(t)
    else:
        conflicting.append(t)

print(f'  - Identik  (anotasi sama): {len(identical)}')
print(f'  - Konflik  (anotasi beda): {len(conflicting)}')

if conflicting:
    print('\nContoh konflik:')
    for t in conflicting[:5]:
        grp = head[head['title'] == t]
        print('  TITLE:', t)
        for sno, row in grp.iterrows():
            print(f'     s_no={sno}: {dict(row["signature"])}')

Title yang muncul di >1 headline: 67
  - Identik  (anotasi sama): 55
  - Konflik  (anotasi beda): 12

Contoh konflik:
  TITLE: Nifty50 in a no-trade zone due to holiday-truncated week: Sandeep Wagle
     s_no=1147: {'Nifty50': 'neutral'}
     s_no=4262: {'Nifty50': 'negative'}
  TITLE: Post RBI policy: Bonds draw comfort from future guidance, rupee steady
     s_no=1154: {'RBI': 'neutral', 'rupee': 'neutral'}
     s_no=6576: {'RBI': 'neutral', 'rupee': 'neutral', 'Bonds': 'neutral'}
  TITLE: Huge activity in Nifty December series surprises market
     s_no=1612: {'Nifty': 'positive'}
     s_no=7281: {'Nifty': 'positive', 'market': 'neutral'}
  TITLE: Midcaps important part of our portfolio: Kunj Bansal, Centrum Wealth Management
     s_no=2626: {'Midcaps': 'positive', 'Centrum Wealth Management': 'neutral'}
     s_no=8152: {'Centrum Wealth Management': 'neutral'}
  TITLE: Crude oil futures down on weak Asian cues
     s_no=3499: {'Crude oil futures': 'negative'}
     s_no=8991: {'Crude

## 2. Dedup

Dua kebijakan berbeda untuk dua jenis duplikat:
- **Konflik (12 title):** buang **semua** s_no-nya — anotasi bertentangan tidak bisa dipercaya.
- **Identik (55 title):** keep s_no terkecil (kemunculan pertama), buang sisanya.

In [3]:
# Drop SEMUA s_no dari title yang konflik
conflict_titles = set(conflicting)
n_conflict_heads = len(head[head['title'].isin(conflict_titles)])

# Untuk identik: keep s_no terkecil per title
non_conflict = head[~head['title'].isin(conflict_titles)]
keep_sno = set(
    non_conflict.reset_index().sort_values('s_no').drop_duplicates('title', keep='first')['s_no']
)

before = long_df['s_no'].nunique()
long_df = long_df[long_df['s_no'].isin(keep_sno)].reset_index(drop=True)
after = long_df['s_no'].nunique()
print(f'Headlines: {before} -> {after}  (dibuang {before - after})')
print(f'  konflik ({len(conflicting)} title, {n_conflict_heads} headlines): dibuang semua')
print(f'  identik ({len(identical)} title): keep s_no terkecil, buang sisanya')
print(f'Pasangan (headline, entity): {len(long_df)}')

Headlines: 10753 -> 10674  (dibuang 79)
  konflik (12 title, 24 headlines): dibuang semua
  identik (55 title): keep s_no terkecil, buang sisanya
Pasangan (headline, entity): 14311


## 3. Stratified split di level headline (80/10/10)

`stratify` pakai majority-sentiment per headline (3 kelas -> aman). Lalu kita **verifikasi distribusi sentimen entity-level** di tiap split harus mirip global.

In [4]:
# Majority sentiment per headline (tie -> kelas paling sering muncul lebih dulu)
maj = long_df.groupby('s_no')['sentiment'].agg(lambda s: s.value_counts().idxmax())
snos, y = maj.index.values, maj.values

train_sno, temp_sno, y_tr, y_tmp = train_test_split(
    snos, y, test_size=0.20, stratify=y, random_state=42)
val_sno, test_sno, _, _ = train_test_split(
    temp_sno, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

split_of = {s: 'train' for s in train_sno}
split_of.update({s: 'val' for s in val_sno})
split_of.update({s: 'test' for s in test_sno})
long_df['split'] = long_df['s_no'].map(split_of)

print('Headlines per split:')
print(long_df.groupby('split')['s_no'].nunique())
print('\nDistribusi sentimen (entity-level) per split:')
print(pd.crosstab(long_df['split'], long_df['sentiment'], normalize='index').round(3))

Headlines per split:
split
test     1068
train    8539
val      1067
Name: s_no, dtype: int64

Distribusi sentimen (entity-level) per split:
sentiment  negative  neutral  positive
split                                 
test          0.265    0.378     0.356
train         0.266    0.380     0.354
val           0.260    0.398     0.342


## 4. NER: word-level BIO tagging (opsi A)

**Tokenisasi:** split whitespace (`\S+`), simpan offset karakter tiap token.

**Penandaan:** boundary check char-level diperluas untuk menangani dua kasus:
1. Batas kanan standar: karakter bukan alnum (mencegah `Gold` ter-tag di `Golden`).
2. **Possessive tanpa apostrof** (`HULs`, `PMC Infratechs`): izinkan `s` + non-alnum di kanan — umum di news headlines.

Semua kemunculan entity di-tag (bukan hanya pertama).

**Validasi:** per-entity — apakah entity tersebut berhasil di-tag setidaknya satu kali? Headline dengan entity yang tidak ter-tag = mismatch, dikecualikan dari NER training (tetap masuk ABSA).

In [5]:
def tokenize_with_spans(text):
    return [(m.group(), m.start(), m.end()) for m in re.finditer(r'\S+', text)]

def find_entity_charspans(title, entity):
    spans = []
    for m in re.finditer(re.escape(entity), title, re.IGNORECASE):
        a, b = m.start(), m.end()
        left  = (a == 0) or (not title[a-1].isalnum())
        right = (b == len(title)) or (not title[b].isalnum())
        # Possessive tanpa apostrof: entity + 's' + non-alnum (e.g. "HULs", "PMC Infratechs")
        if not right and title[b].lower() == 's':
            after_s = b + 1
            right = (after_s >= len(title)) or (not title[after_s].isalnum())
        if left and right:
            spans.append((a, b))
    return spans

def bio_tag(title, entities):
    toks = tokenize_with_spans(title)
    tags = ['O'] * len(toks)
    untagged = []
    for entity in set(entities):
        tagged_once = False
        for (a, b) in find_entity_charspans(title, entity):
            members = [i for i, (_, s, e) in enumerate(toks) if s < b and e > a]
            if not members or any(tags[i] != 'O' for i in members):
                continue
            tags[members[0]] = 'B-ENT'
            for i in members[1:]:
                tags[i] = 'I-ENT'
            tagged_once = True
            # Tidak break: tag SEMUA kemunculan entity di title
        if not tagged_once:
            untagged.append(entity)
    return [t for t, _, _ in toks], tags, untagged

In [6]:
ner_rows, mismatches = [], []
for sno, g in long_df.groupby('s_no'):
    title = g['title'].iloc[0]
    entities = list(g['entity'])
    tokens, tags, untagged = bio_tag(title, entities)
    ner_rows.append({'s_no': int(sno), 'split': g['split'].iloc[0],
                     'tokens': tokens, 'ner_tags': tags,
                     'has_mismatch': len(untagged) > 0})
    if untagged:
        mismatches.append((sno, title, entities, untagged, tags))

ner_df = pd.DataFrame(ner_rows)
n_clean = (~ner_df['has_mismatch']).sum()
print(f'Headlines ter-tag: {len(ner_df)}')
print(f'  bersih (semua entity ter-tag): {n_clean}')
print(f'  mismatch (ada entity tak ter-tag): {len(mismatches)}')
if mismatches:
    print('\nContoh mismatch (genuine annotation inconsistency):')
    for sno, title, ents, untagged, tags in mismatches[:8]:
        print(f'  [{sno}] {title}')
        print(f'        entities={ents}  untagged={untagged}')

Headlines ter-tag: 10674
  bersih (semua entity ter-tag): 10635
  mismatch (ada entity tak ter-tag): 39

Contoh mismatch (genuine annotation inconsistency):
  [141] Expert's take: Rupee-dollar pair to test 45.30-45.50 levels
        entities=['Rupee', 'dollar']  untagged=['dollar']
  [221] Nifty50 to find support at 7,950-7,930: Mitesh Thacker
        entities=['Nifty']  untagged=['Nifty']
  [351] BoR awaits legal view for ICICI-BoR merger
        entities=['ICICI', 'BoR']  untagged=['ICICI']
  [793] Alibaba's record IPO covered entire deal in 2 days: Sources
        entities=['Alibab']  untagged=['Alibab']
  [812] Centre seeks more time to pass FTIL-NSEL final merger order
        entities=['FTIL', 'NSEL']  untagged=['FTIL']
  [913] More time required for order on NSEL-FTIL merger, says Government
        entities=['NSEL', 'FTIL']  untagged=['FTIL']
  [1050] Corporate affairs ministry finalises share swap ratio for proposed NSEL-FTIL merger
        entities=['NSEL', 'FTIL']  untagged=

In [7]:
# Sanity: tampilkan satu contoh multi-entity hasil tagging
ex = ner_df[ner_df['ner_tags'].apply(lambda t: t.count('B-ENT') >= 2)].iloc[0]
print('Contoh tagging multi-entity (s_no =', ex['s_no'], '):')
for tok, tag in zip(ex['tokens'], ex['ner_tags']):
    print(f'  {tok:<18} {tag}')

Contoh tagging multi-entity (s_no = 9 ):
  Gold               B-ENT
  shines             O
  on                 O
  seasonal           O
  demand;            O
  Silver             B-ENT
  dull               O


## 5. ABSA format

Satu baris = satu pasangan `(title, entity)` + label sentimen integer. Mapping: `negative=0, neutral=1, positive=2`. Format input untuk model nanti dibentuk saat training (`[CLS] title [SEP] entity [SEP]`), bukan di sini -- di sini kita simpan komponen mentahnya saja.

In [8]:
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
long_df['label'] = long_df['sentiment'].map(LABEL2ID)
assert long_df['label'].notna().all(), 'Ada sentimen di luar 3 kelas!'

absa_cols = ['s_no', 'title', 'entity', 'sentiment', 'label', 'split']
absa_df = long_df[absa_cols].copy()
print('ABSA pairs:', len(absa_df))
print(absa_df.groupby('split').size())
absa_df.head()

ABSA pairs: 14311
split
test      1448
train    11433
val       1430
dtype: int64


,s_no,title,entity,sentiment,label,split
0,1,SpiceJet to issue 6.4 crore warrants to promoters,SpiceJet,neutral,1,val
1,2,MMTC Q2 net loss at Rs 10.4 crore,MMTC,neutral,1,train
2,3,"Mid-cap funds can deliver more, stay put: Experts",Mid-cap funds,positive,2,train
3,4,Mid caps now turn into market darlings,Mid caps,positive,2,test
4,5,"Market seeing patience, if not conviction: Pra...",Market,neutral,1,train


## 6. Simpan output per split

NER -> JSONL (1 headline per baris). ABSA -> CSV (1 pasangan per baris).

In [9]:
def save_jsonl(df, path):
    with open(path, 'w', encoding='utf-8') as f:
        for _, r in df.iterrows():
            f.write(json.dumps({'s_no': r['s_no'],
                                'tokens': r['tokens'],
                                'ner_tags': r['ner_tags']}, ensure_ascii=False) + '\n')

# NER: hanya pakai headline yang semua entity-nya ter-tag
ner_clean = ner_df[~ner_df['has_mismatch']]
print(f'NER training: {len(ner_clean)} headlines '
      f'({len(mismatches)} mismatch dikecualikan dari NER, tetap ada di ABSA)\n')

for split in ['train', 'val', 'test']:
    n_path = DATA_PROCESSED / f'ner_{split}.jsonl'
    a_path = DATA_PROCESSED / f'absa_{split}.csv'
    save_jsonl(ner_clean[ner_clean['split'] == split], n_path)
    absa_df[absa_df['split'] == split].drop(columns='split').to_csv(a_path, index=False, encoding='utf-8')
    print(f'{split:5s} -> NER {n_path.name} ({(ner_clean["split"]==split).sum()} headlines), '
          f'ABSA {a_path.name} ({(absa_df["split"]==split).sum()} pairs)')

print('\nSelesai. Semua file di', DATA_PROCESSED)

NER training: 10635 headlines (39 mismatch dikecualikan dari NER, tetap ada di ABSA)

train -> NER ner_train.jsonl (8509 headlines), ABSA absa_train.csv (11433 pairs)
val   -> NER ner_val.jsonl (1063 headlines), ABSA absa_val.csv (1430 pairs)
test  -> NER ner_test.jsonl (1063 headlines), ABSA absa_test.csv (1448 pairs)

Selesai. Semua file di D:\Text Mining Projects\Financial-News-Miner\data\processed


## 7. Ringkasan Sprint 2 (isi setelah run ulang)

- [ ] Konflik dibuang: ___ title (___ headlines); identik dedup: ___ headline dibuang
- [ ] Headlines setelah dedup: ___
- [ ] Split: train ___ / val ___ / test ___
- [ ] Distribusi sentimen antar-split konsisten? (ya/tidak)
- [ ] BIO mismatch setelah fix possessive: ___ (dari 122 semula)
- [ ] NER training headlines: ___ (clean) / ABSA pairs tetap: ___
- [ ] File output tersimpan: 3 NER jsonl + 3 ABSA csv